In [ ]:
import subprocess, sys

for pkg in ["openai-agents", "gradio", "python-dotenv", "mcp", "httpx"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )


In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from contextlib import AsyncExitStack
import asyncio, os, sys, time

load_dotenv(override=True)
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
assert os.getenv("SERPER_API_KEY"), "Missing SERPER_API_KEY"


In [ ]:
venv_bin = os.path.dirname(sys.executable)
sandbox_path = os.path.abspath("sandbox")
os.makedirs(sandbox_path, exist_ok=True)

serper_params = {
    "command": sys.executable,
    "args": [os.path.join(os.getcwd(), "serper_server.py")],
    "env": {**os.environ, "SERPER_API_KEY": os.getenv("SERPER_API_KEY")},
}

fetch_params = {
    "command": os.path.join(venv_bin, "mcp-server-fetch"),
    "args": [],
}

files_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path],
}

TIMEOUT = 60
MODEL = "gpt-4o-mini"


In [ ]:
agents_config = {
    "scout": {
        "label": "Scout",
        "status_text": "Finding competitors",
        "instructions": """Find the top 5 direct competitors for the given company.
For each one give me: name, where they're based, what they do, and why they're a threat.
Search Google for things like "[company] competitors", "[company] vs", "[company] alternatives".
Keep it to a clean numbered list, no fluff."""
    },
    "product": {
        "label": "Product Analyst",
        "status_text": "Researching products & pricing",
        "instructions": """Dig into what each competitor actually sells.
I need: their main products, how they charge (free tier? subscriptions? enterprise pricing?),
what makes them different from our target company, any tech advantages, and anything they
launched recently. Hit up their pricing pages with Fetch if you can find them."""
    },
    "strategy": {
        "label": "Strategy Analyst",
        "status_text": "Analyzing strategies",
        "instructions": """Figure out each competitor's playbook. How do they make money,
who are they selling to, how are they growing (M&A, partnerships, new markets)?
Look for recent news -- funding rounds, layoffs, pivots, big announcements.
Focus on the last year. I want specific dates and numbers, not vague statements."""
    },
    "swot": {
        "label": "SWOT Analyst",
        "status_text": "Building SWOT matrix",
        "instructions": """Build a SWOT analysis for our target company compared to its competitors.
Strengths (what do we do better), Weaknesses (where competitors beat us),
Opportunities (gaps we could exploit), Threats (what should worry us).
Give me 4-6 solid points per section backed by what you actually found in your research."""
    },
    "director": {
        "label": "Intel Director",
        "status_text": "Writing final report",
        "instructions": """You get reports from three analysts. Pull it all together into one
Competitive Intelligence Report. I want: executive summary, competitor profiles,
a product/pricing comparison table, the strategic landscape, SWOT matrix,
and 5-7 concrete recommendations. Save as intel_report.md in the sandbox.
This should be thorough -- at least 2000 words."""
    },
}

In [ ]:
pipeline_order = ["scout", "product", "strategy", "swot", "director"]


def status_md(done, active, t0):
    lines = [f"### Pipeline &nbsp; `{time.time() - t0:.0f}s`\n"]
    for key in pipeline_order:
        cfg = agents_config[key]
        if key in done:
            lines.append(f"- &#x2705; **{cfg['label']}** -- {cfg['status_text']} *({done[key]:.1f}s)*")
        elif key == active:
            lines.append(f"- &#x1F50D; **{cfg['label']}** -- {cfg['status_text']} *(running...)*")
        else:
            lines.append(f"- &#x23F3; **{cfg['label']}** -- {cfg['status_text']}")
    return "\n".join(lines)


async def dispatch(name, prompt, servers):
    cfg = agents_config[name]
    agent = Agent(name=name, instructions=cfg["instructions"], model=MODEL, mcp_servers=servers)
    t0 = time.time()
    with trace(name):
        result = await Runner.run(agent, prompt, max_turns=30)
    return result.final_output, time.time() - t0


async def run_pipeline(company, context=""):
    t0 = time.time()
    done = {}
    query = f"Company: {company}"
    if context:
        query += f"\nIndustry context: {context}"

    yield "**Spinning up MCP servers...**", status_md(done, "scout", t0)

    stack = AsyncExitStack()
    try:
        serper = await stack.enter_async_context(MCPServerStdio(params=serper_params, client_session_timeout_seconds=TIMEOUT))
        fetch = await stack.enter_async_context(MCPServerStdio(params=fetch_params, client_session_timeout_seconds=TIMEOUT))
        files = await stack.enter_async_context(MCPServerStdio(params=files_params, client_session_timeout_seconds=TIMEOUT))

        search = [serper, fetch]

        scout_out, dt = await dispatch("scout", f"Find the top 5 competitors for: {query}", search)
        done["scout"] = dt
        yield f"**Competitors found. Analysts deploying...**\n\n{scout_out}", status_md(done, "product", t0)

        briefing = f"{query}\n\nCompetitors:\n{scout_out}"

        async def analyst(name, task):
            r, dt = await dispatch(name, f"{task}\n\n{briefing}", search)
            done[name] = dt
            return r

        prod, strat, swot = await asyncio.gather(
            analyst("product", "Analyze products and pricing."),
            analyst("strategy", "Analyze competitive strategies."),
            analyst("swot", "Build a SWOT analysis."),
        )
        yield "**Analysts done. Director compiling report...**", status_md(done, "director", t0)

        director_brief = f"Target: {query}\n\nPRODUCT:\n{prod}\n\nSTRATEGY:\n{strat}\n\nSWOT:\n{swot}\n\nSave as intel_report.md."
        final, dt = await dispatch("director", director_brief, [serper, fetch, files])
        done["director"] = dt
        yield final, status_md(done, None, t0)

    except Exception as e:
        yield f"**Error:** {e}", status_md(done, None, t0)
    finally:
        await stack.aclose()

In [ ]:
import gradio as gr

async def analyze(company, context):
    if not company.strip():
        yield "Enter a company name first.", "### Pipeline\n\nWaiting..."
        return
    try:
        async for report, st in run_pipeline(company.strip(), context.strip()):
            yield report, st
    except Exception as e:
        yield f"**Error:** {e}", f"### Pipeline\n\nFailed: {e}"

with gr.Blocks(title="Competitor Intel", theme=gr.themes.Soft()) as ui:
    gr.Markdown("# Competitor Intel Agent")
    with gr.Row():
        with gr.Column(scale=2):
            inp_company = gr.Textbox(label="Company", placeholder="e.g. Safaricom, Stripe, Notion...")
            inp_context = gr.Textbox(label="Industry context (optional)", placeholder="e.g. mobile payments in East Africa")
            btn = gr.Button("Go", variant="primary", size="lg")
        with gr.Column(scale=1):
            status_panel = gr.Markdown("### Pipeline\n\nReady.")
    gr.Markdown("---")
    report_panel = gr.Markdown("*Report will appear here.*")
    btn.click(fn=analyze, inputs=[inp_company, inp_context], outputs=[report_panel, status_panel])

ui.launch()